<a href="https://colab.research.google.com/github/ArshanBhanage/Alt-Pay/blob/main/RAG_architecture_to_answer_State_of_Food_Security_and_Nutrition_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pymupdf pdfplumber chromadb sentence-transformers rank_bm25 anthropic unidecode tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.0 MB/s eta 

In [2]:
import os
from pathlib import Path
from typing import List, Dict, Any
import re
from dataclasses import dataclass


import fitz # PyMuPDF
import pdfplumber
import numpy as np
from tqdm import tqdm
from unidecode import unidecode


import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi


from anthropic import Anthropic

In [21]:
PERSIST_DIR = "/content/sofi_index"
COLLECTION_NAME = "sofi"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

CLAUDE_MODEL = "claude-sonnet-4-20250514"
MAX_TOKENS = 800

In [6]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_KEY"] = userdata.get("ANTHROPIC_KEY")

if not os.environ.get("ANTHROPIC_KEY"):
  print("\n[Action Required] Set your Anthropic API key in Colab secrets as 'ANTHROPIC_KEY'.\n")
else:
  client = Anthropic(api_key=os.environ.get("ANTHROPIC_KEY"))
  print("Anthropic client initialized successfully.")

Anthropic client initialized successfully.


In [9]:
PDF_2023 = Path("Recommender_Systems/SOFI-2023.pdf")
PDF_2024 = Path("Recommender_Systems/SOFI-2024.pdf")


assert PDF_2023.exists(), "SOFI-2023.pdf not found in working directory."
assert PDF_2024.exists(), "SOFI-2024.pdf not found in working directory."

In [47]:
HEADING_RE = re.compile(r"^\s*(\d+(?:\.\d+)*)\s+.+", re.MULTILINE)


@dataclass
class DocChunk:
  content: str
  meta: Dict[str, Any]




def normalize_text(text: str) -> str:
  text = unidecode(text)
  text = re.sub(r"\s+", " ", text).strip()
  return text




def extract_text_pymupdf(pdf_path: Path) -> List[DocChunk]:
  doc = fitz.open(pdf_path)
  chunks = []
  for i, page in enumerate(doc, start=1):
    text = page.get_text("text") or ""
    text = normalize_text(text)
    section_match = HEADING_RE.search(text)
    section = section_match.group(0).strip() if section_match else "Unlabeled"
    chunks.append(DocChunk(content=text, meta={"page": i, "section": section}))
  return chunks




def split_into_windows(text: str, max_tokens: int = 1200, overlap: int = 180) -> List[str]:
  words = text.split()
  if not words:
    return []
  step = max(1, max_tokens - overlap)
  out = []
  for start in range(0, len(words), step):
    out.append(" ".join(words[start:start + max_tokens]))
  return out




def chunk_pages(pages: List[DocChunk], source_label: str) -> List[DocChunk]:
  out = []
  for p in pages:
    windows = split_into_windows(p.content, max_tokens=1200, overlap=180)
  for w in windows:
    if not w.strip():
      continue
    out.append(DocChunk(
    content=w,
    meta={"source": source_label, "page": p.meta["page"], "section": p.meta.get("section", "Unlabeled")}
    ))
  return out

In [28]:
print("\nParsing PDFs (this may take ~1–3 minutes)...")
raw_2023 = extract_text_pymupdf(PDF_2023)
raw_2024 = extract_text_pymupdf(PDF_2024)

docs_2023 = chunk_pages(raw_2023, "SOFI-2023")
docs_2024 = chunk_pages(raw_2024, "SOFI-2024")
docs = docs_2023 + docs_2024

print(f"Total base pages: {len(raw_2023) + len(raw_2024)}; total chunks: {len(docs)}")

corpus_texts = [d.content for d in docs]
corpus_tokens = [t.split() for t in corpus_texts]
bm25 = BM25Okapi(corpus_tokens)

embedder = SentenceTransformer(EMBED_MODEL_NAME)
client_db = chromadb.Client(Settings(is_persistent=True, persist_directory=PERSIST_DIR))

try:
    client_db.delete_collection(COLLECTION_NAME)
except Exception:
    pass
col = client_db.create_collection(COLLECTION_NAME)

print("Computing embeddings (this may take a few minutes)...")
embs = embedder.encode(corpus_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
col.add(
    ids=[str(i) for i in range(len(docs))],
    embeddings=embs.tolist(),
    documents=corpus_texts,
    metadatas=[d.meta for d in docs],
)

print("Loading reranker...")
reranker = CrossEncoder(RERANKER_NAME)


Parsing PDFs (this may take ~1–3 minutes)...
Total base pages: 602; total chunks: 603
Computing embeddings (this may take a few minutes)...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Loading reranker...


In [48]:
from typing import Tuple

def hybrid_search(query: str, k_sparse: int = 60, k_dense: int = 60, k_final: int = 10) -> List[DocChunk]:
    sparse_docs = bm25.get_top_n(query.split(), corpus_texts, n=k_sparse)
    ids_sparse = []
    seen = set()
    for t in sparse_docs:
        idx = corpus_texts.index(t)
        if idx not in seen:
            ids_sparse.append(idx)
            seen.add(idx)


    q_emb = embedder.encode([query], convert_to_numpy=True)
    res = col.query(query_embeddings=q_emb.tolist(), n_results=min(k_dense, len(docs)))
    ids_dense = [int(_id) for _id in res["ids"][0]] if res.get("ids") else []

    cand_ids = list(dict.fromkeys(ids_sparse + ids_dense))

    pairs = [[query, corpus_texts[i]] for i in cand_ids]
    if not pairs:
        return []
    rr = reranker.predict(pairs)
    order = np.argsort(-np.array(rr))[:k_final]
    selected = [cand_ids[i] for i in order]

    results = []
    for idx in selected:
        results.append(DocChunk(
            content=docs[idx].content,
            meta=docs[idx].meta
        ))
    return results

In [49]:
ANSWER_INSTRUCTIONS = (
    "You answer questions about the FAO/UN State of Food Security and Nutrition (SOFI) reports.\n"
    "Use ONLY the provided context.\n"
    "Cite sources at the end of each bullet or paragraph like (SOFI-2024, p. 132) or (SOFI-2023, p. 87).\n"
    "If the answer is not supported by the context, say you don't know. Keep temperature low."
)

def extract_text_pymupdf(pdf_path: Path) -> List[DocChunk]:
    doc = fitz.open(pdf_path)
    chunks = []
    for page in doc:
        pdf_idx = page.number + 1
        label = page.get_label() or str(pdf_idx)
        text = normalize_text(page.get_text("text") or "")
        m = HEADING_RE.search(text)
        section = m.group(0).strip() if m else "Unlabeled"
        chunks.append(DocChunk(
            content=text,
            meta={"page_pdf": pdf_idx, "page_label": label, "section": section}
        ))
    return chunks

def chunk_pages(pages: List[DocChunk], source_label: str) -> List[DocChunk]:
    out = []
    for p in pages:
        windows = split_into_windows(p.content, max_tokens=1200, overlap=180)
        for w in windows:
            if not w.strip():
                continue
            out.append(DocChunk(
                content=w,
                meta={
                    "source": source_label,
                    "page_pdf": p.meta["page_pdf"],
                    "page_label": p.meta["page_label"],
                    "section": p.meta.get("section", "Unlabeled"),
                }
            ))
    return out

def format_cite(meta: dict) -> str:
    src = meta.get("source", "SOFI")
    label = meta.get("page_label") or str(meta.get("page_pdf") or meta.get("page") or "?")
    pdf_p = meta.get("page_pdf") or meta.get("page")
    return f"{src}, p. {label}" + (f" (PDF p.{pdf_p})" if pdf_p else "")

def build_prompt(query: str, contexts: List[DocChunk]) -> str:
    ctx_blocks, whitelist = [], []
    for c in contexts:
        cite = format_cite(c.meta)
        ctx_blocks.append(f"[{cite}] {c.content}")
        whitelist.append(cite)
    context_str = "\n\n".join(ctx_blocks)
    allowed = ", ".join(sorted(set(whitelist)))
    return (
        f"{ANSWER_INSTRUCTIONS}\n\n"
        f"Context:\n{context_str}\n\n"
        f"Allowed citations (must only cite from this list): {allowed}\n"
        f"Question: {query}\n\n"
        f"Answer with bullet points. After each bullet, add exactly one citation from the allowed list."
    )



def answer_with_claude(query: str, k_ctx: int = 12) -> str:
    ctx = hybrid_search(query, k_sparse=100, k_dense=100, k_final=k_ctx)
    if not ctx:
        return "No relevant context found in the indexed documents."
    prompt = build_prompt(query, ctx)
    resp = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=MAX_TOKENS,
        temperature=0.1,
        messages=[{"role": "user", "content": prompt}],
        system="Ground your answers ONLY in the provided context and cite pages from the allowed list."
    )
    parts = [b.text for b in resp.content if getattr(b, "type", "") == "text"]
    return "".join(parts).strip()

In [50]:
print("\nIndex ready. Try a quick retrieval-only check:")
q_demo = "List major reasons for food insecurity in 2024."
ctx_demo = hybrid_search(q_demo, k_final=5)
for i, c in enumerate(ctx_demo, 1):
    print(f"\n[{i}] {c.meta['source']} p.{c.meta['page']} :: {c.meta['section']}")
    print(c.content[:240], '...')


Index ready. Try a quick retrieval-only check:

[1] SOFI-2024 p.117 :: Unlabeled
THE STATE OF FOOD SECURITY AND NUTRITION IN THE WORLD 2024 FIGURE 27 FOOD AVAILABILITY, HEALTH SERVICES AND ENVIRONMENTAL HEALTH, AND CONFLICT AND INEQUALITY TAKE THE MAJORITY OF THE OFFICIAL DEVELOPMENT ASSISTANCE AND OTHER OFFICIAL FLOWS  ...

[2] SOFI-2024 p.43 :: Unlabeled
THE STATE OF FOOD SECURITY AND NUTRITION IN THE WORLD 2024 BOX 2 DEEPENING HUMANITARIAN CRISES INCREASE ACUTE FOOD INSECURITY AND THREATEN THE RIGHT TO ADEQUATE FOOD IN MANY PLACES IN THE WORLD During the preparation of this edition of The  ...

[3] SOFI-2024 p.181 :: Unlabeled
THE STATE OF FOOD SECURITY AND NUTRITION IN THE WORLD 2024 millions will have been pushed into crisis or worse levels of acute food insecurity, and insufficient progress will have been made to meet all global nutrition targets. The resultin ...

[4] SOFI-2024 p.96 :: Unlabeled
CHAPTER 3 A NEW DEFINITION OF FINANCING FOR FOOD SECURITY AND NUTRITION BOX 8 PROTR

In [51]:
Q1 = "List major reasons for food insecurity in 2024. Provide page-cited bullets."
Q2 = "Explain forms, drivers, and consequences of malnutrition in conflict/war zones, with examples and page citations."
Q3 = "Explain how increased food prices affect food security (availability, access, utilization, stability) with page citations."

print(answer_with_claude(Q1))
# print(answer_with_claude(Q2))
# print(answer_with_claude(Q3))

Based on the provided context, here are the major reasons for food insecurity in 2024:

• **Conflict and humanitarian crises** - Deepening humanitarian crises continued to seriously erode food security, with nearly 282 million people facing high levels of acute food insecurity in 59 food-crisis countries/territories in 2023 (SOFI-2024, p. 43)

• **Climate variability and extremes** - Climate extremes are identified as one of the major drivers increasing in frequency and intensity, affecting food security and nutrition worldwide (SOFI-2024, p. 20)

• **Economic slowdowns and downturns** - Economic downturns continue to be a major driver of food insecurity, with countries affected by economic downturns experiencing notably sharper increases in undernourishment between 2019 and 2023 (SOFI-2024, p. 96)

• **Lack of access to and unaffordability of healthy diets** - The unaffordability of nutritious foods remains a critical underlying structural factor impeding food security achievement (SO

In [52]:
Q1 = "List major reasons for food insecurity in 2024. Provide page-cited bullets."
Q2 = "Explain forms, drivers, and consequences of malnutrition in conflict/war zones, with examples and page citations."
Q3 = "Explain how increased food prices affect food security (availability, access, utilization, stability) with page citations."

# print(answer_with_claude(Q1))
print(answer_with_claude(Q2))
# print(answer_with_claude(Q3))

Based on the provided context, here are the forms, drivers, and consequences of malnutrition in conflict/war zones:

## Forms of Malnutrition in Conflict Zones

• **Multiple forms of malnutrition occur simultaneously** - including hunger, food insecurity, stunting, wasting, and overweight in children under five years of age, creating complex nutritional challenges in conflict-affected areas (SOFI-2024, p. 75)

• **Acute food insecurity reaches emergency levels** - with people facing severe hunger that can escalate to famine conditions, requiring immediate humanitarian intervention (SOFI-2024, p. 130)

## Major Drivers of Malnutrition in Conflict Areas

• **Conflict is identified as one of the primary major drivers** - along with climate variability and extremes, and economic slowdowns and downturns, that intensify food insecurity and malnutrition (SOFI-2024, p. 20)

• **Multiple drivers compound impacts** - countries affected by conflict often face additional challenges from climate ex

In [53]:
Q1 = "List major reasons for food insecurity in 2024. Provide page-cited bullets."
Q2 = "Explain forms, drivers, and consequences of malnutrition in conflict/war zones, with examples and page citations."
Q3 = "Explain how increased food prices affect food security (availability, access, utilization, stability) with page citations."

# print(answer_with_claude(Q1))
# print(answer_with_claude(Q2))
print(answer_with_claude(Q3))

Based on the provided context, here's how increased food prices affect food security across its four dimensions:

## Food Access
• Higher food prices directly erode household income gains and worsen access to food, particularly affecting the poorest segments of the population who spend a larger share of their income on food (SOFI-2023, p. 44)

• Food price increases create economic challenges related to the high cost of nutritious foods, with poorer households prioritizing meeting dietary energy requirements over nutritional quality by spending resources on more affordable foods that tend to be of high energy density and minimal nutritional value (SOFI-2023, p. 94)

## Food Availability
• Agricultural production can suffer from food price inflation effects, as seen in regions where the surge in international fertilizer prices combined with severe droughts and floods negatively impacted agricultural output (SOFI-2023, p. 45)

• Food price increases can disrupt food supplies, markets and